# Conversational Threads

Many LLM applications have a chatbot-like interface in which the user and the LLM application engage in a multi-turn conversation. In order to track these conversations, you can use the Threads feature in LangSmith.

This is relevant to our RAG application, which should maintain context from prior conversations with users.

### Setup

In [1]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv()

True

### Group traces into threads


A Thread is a sequence of traces representing a single conversation. Each response is represented as its own trace, but these traces are linked together by being part of the same thread.

To associate traces together, you need to pass in a special metadata key where the value is the unique identifier for that thread.

The key value is the unique identifier for that conversation. The key name should be one of:

- session_id
- thread_id
- conversation_id.

The value should be a UUID.

In [2]:
import uuid
thread_id = uuid.uuid4()

In [3]:
import os
import nest_asyncio
from typing import List
from openai import OpenAI
from langsmith import traceable
from utils import get_vector_db_retriever

MODEL_NAME = "openai/gpt-oss-120b"

# Route standard OpenAI client to Groq
openai_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

nest_asyncio.apply()
retriever = get_vector_db_retriever()


@traceable(run_type="chain")
def retrieve_documents(question: str):
    return retriever.invoke(question)


@traceable(run_type="chain")
def generate_response(question: str, documents):
    # Rule 5: Token Diet [:600]
    formatted_docs = "\n\n".join(doc.page_content[:600] for doc in documents)
    rag_system_prompt = """You are an assistant for question-answering tasks. 
    Use the following pieces of retrieved context to answer the latest question in the conversation. 
    If you don't know the answer, just say that you don't know. 
    Use three sentences maximum and keep the answer concise.
    """
    messages = [
        {"role": "system", "content": rag_system_prompt},
        {"role": "user", "content": f"Context: {formatted_docs} \n\n Question: {question}"}
    ]
    return call_openai(messages)


@traceable(run_type="llm")
def call_openai(
    messages: List[dict], model: str = MODEL_NAME, temperature: float = 0.0
) -> str:
    return openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    ) 

@traceable(run_type="chain")
def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content

[1/3] 🔍 Locating union.parquet...
[2/3] 📖 Found 'resources\union.parquet'. Reading parquet into DataFrame...
      Loaded 2187 document chunks from parquet.
      Using text column: 'texts', metadata column: 'metadatas'
[3/3] ⚡ Building local TF-IDF index across 2187 documents...
✅ Retriever ready in 2.42 seconds! Zero API calls, zero web scraping.


### Now let's run our application twice with this thread_id

In [4]:
question = "How do I add metadata to a Trace?"
ai_answer = langsmith_rag(question, langsmith_extra={"metadata": {"thread_id": thread_id}})
print(ai_answer)

Add a dictionary of key‑value pairs as the `metadata` parameter when you create or update a trace (e.g., `client.create_trace(..., metadata={"env":"prod","version":"1.2"})`). You can also edit an existing trace in the LangSmith UI and enter metadata in the “Metadata” section. The metadata will then be stored with that trace and visible in searches and filters.


In [5]:
question = "How can I add tags to a Trace?"
ai_answer = langsmith_rag(question, langsmith_extra={"metadata": {"thread_id": thread_id}})
print(ai_answer)

Add tags by passing a list of strings in the `tags` field of the `RunnableConfig` when you run a chain or runnable. For example:

```python
chain.invoke(
    {"question": "…"},
    config={"tags": ["my-tag", "experiment-1"]},
)
```

The tags will be attached to the trace and appear in the LangSmith UI for filtering and grouping.


### Let's take a look in LangSmith!